In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
from skimage import data, io, color, filters, segmentation, measure, morphology
from skimage.filters import threshold_otsu, gaussian
from skimage.segmentation import clear_border
from skimage.measure import label, regionprops
from skimage.color import rgb2hsv, rgb2gray
import pandas as pd
import os

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['image.interpolation'] = 'nearest'

# RUTA DE LA IMAGEN A PROCESAR
RUTA_IMAGEN = './image.jpg'
print(f'Imagen a procesar: {RUTA_IMAGEN}')
print(f'¿Existe?: {os.path.exists(RUTA_IMAGEN)}')


# DETECCIÓN DE COLORES EN REGIONES (SPAGHETTI)

In [ ]:
# ============================================
# FUNCIÓN: DETECTAR COLORES EN REGIONES
# Recibe ruta de imagen y detecta cada color
# separándolo en regiones etiquetadas
# ============================================

def detectar_colores_en_regiones(ruta_imagen):
    
    # PASO 1: Cargar imagen desde ruta
    if not os.path.exists(ruta_imagen):
        print(f'ERROR: No se encontró imagen en {ruta_imagen}')
        return None, None
    
    img = io.imread(ruta_imagen)
    
    # Convertir a RGB si es necesario
    if img.shape[-1] == 4:
        img = color.rgba2rgb(img)
    
    print(f'PASO 1: Imagen cargada desde {ruta_imagen}')
    print(f'  Forma: {img.shape}')
    
    # PASO 2: Convertir a HSV para detección de color
    img_hsv = rgb2hsv(img)
    print('PASO 2: Convertido a HSV')
    
    # PASO 3: Definir rangos de colores (H, S, V) con tolerancia
    rangos_colores = {
        'ROJO':    {'h': (0.0, 0.05), 's': (0.5, 1.0), 'v': (0.5, 1.0)},
        'AZUL':    {'h': (0.55, 0.65), 's': (0.5, 1.0), 'v': (0.5, 1.0)},
        'VERDE':   {'h': (0.25, 0.35), 's': (0.5, 1.0), 'v': (0.5, 1.0)},
        'AMARILLO':{'h': (0.1, 0.15), 's': (0.6, 1.0), 'v': (0.7, 1.0)},
        'MAGENTA': {'h': (0.8, 0.9), 's': (0.5, 1.0), 'v': (0.6, 1.0)},
        'CYAN':    {'h': (0.45, 0.55), 's': (0.5, 1.0), 'v': (0.6, 1.0)}
    }
    
    resultados = {}
    
    # PASO 4: Para cada color, crear máscara y contar regiones
    for nombre_color, rango in rangos_colores.items():
        
        mascara_h = (img_hsv[:, :, 0] >= rango['h'][0]) & (img_hsv[:, :, 0] <= rango['h'][1])
        mascara_s = (img_hsv[:, :, 1] >= rango['s'][0]) & (img_hsv[:, :, 1] <= rango['s'][1])
        mascara_v = (img_hsv[:, :, 2] >= rango['v'][0]) & (img_hsv[:, :, 2] <= rango['v'][1])
        
        mascara_color = mascara_h & mascara_s & mascara_v
        mascara_uint8 = mascara_color.astype(np.uint8) * 255
        
        etiquetas_color = label(mascara_color)
        num_regiones = etiquetas_color.max()
        regiones = regionprops(etiquetas_color)
        
        resultados[nombre_color] = {
            'mascara': mascara_uint8,
            'num_regiones': num_regiones,
            'regiones': regiones,
            'etiquetas': etiquetas_color
        }
        
        print(f'PASO 4: {nombre_color} -> {num_regiones} regiones detectadas')
    
    return img, resultados


def mostrar_resultados_colores(img, resultados):
    
    num_colores = len(resultados)
    fig, axes = plt.subplots(2, num_colores + 1, figsize=(4 * (num_colores + 1), 8))
    
    axes[0, 0].imshow(img)
    axes[0, 0].set_title('ORIGINAL')
    axes[0, 0].axis('off')
    
    axes[1, 0].axis('off')
    
    idx = 1
    for nombre_color, datos in resultados.items():
        axes[0, idx].imshow(datos['mascara'], cmap='gray')
        axes[0, idx].set_title(f'{nombre_color}\n{datos["num_regiones"]} regiones')
        axes[0, idx].axis('off')
        
        img_aislada = img.copy()
        mascara_bool = datos['mascara'] > 0
        img_aislada[~mascara_bool] = 0
        axes[1, idx].imshow(img_aislada)
        axes[1, idx].set_title(f'{nombre_color} Aislado')
        axes[1, idx].axis('off')
        
        idx += 1
    
    plt.tight_layout()
    plt.show()


def contar_colores_en_region(img, region, resultados):
    y, x = region.bbox[0], region.bbox[1]
    y2, x2 = region.bbox[2], region.bbox[3]
    
    conteo = {}
    
    for nombre_color, datos in resultados.items():
        mascara_region = datos['mascara'][y:y2, x:x2]
        pixeles_color = np.sum(mascara_region > 0)
        total_pixeles = mascara_region.size
        porcentaje = (pixeles_color / total_pixeles) * 100 if total_pixeles > 0 else 0
        
        conteo[nombre_color] = {
            'pixeles': pixeles_color,
            'porcentaje': porcentaje
        }
    
    return conteo


# EJECUTAR DETECCIÓN EN ./image.png

In [ ]:
# ============================================
# LLAMAR FUNCIÓN CON RUTA ./image.png
# ============================================

print('=' * 50)
print('DETECTANDO COLORES EN REGIONES')
print('=' * 50)

img_procesada, datos_colores = detectar_colores_en_regiones(RUTA_IMAGEN)

if img_procesada is not None:
    mostrar_resultados_colores(img_procesada, datos_colores)


# FUNCIÓN SPAGHETTI COMPLETA (Dividir y Contar por Color)

In [ ]:
def dividir_y_contar_por_color(ruta_imagen):
    
    # PASO 1: Cargar imagen desde ruta
    if not os.path.exists(ruta_imagen):
        print(f'ERROR: No se encontró imagen en {ruta_imagen}')
        return None
    
    img = io.imread(ruta_imagen)
    
    if img.shape[-1] == 4:
        img = color.rgba2rgb(img)
    
    print(f'PASO 1: Imagen cargada - forma {img.shape}')
    
    # PASO 2: Convertir a HSV
    img_hsv = rgb2hsv(img)
    print('PASO 2: HSV')
    
    # PASO 3: Definir colores
    colores_definidos = [
        ('ROJO', (0.0, 0.05), (0.5, 1.0), (0.5, 1.0)),
        ('AZUL', (0.55, 0.65), (0.5, 1.0), (0.5, 1.0)),
        ('VERDE', (0.25, 0.35), (0.5, 1.0), (0.5, 1.0)),
        ('AMARILLO', (0.1, 0.15), (0.6, 1.0), (0.7, 1.0))
    ]
    
    # Crear figura
    fig, axes = plt.subplots(3, 5, figsize=(20, 12))
    
    axes[0, 0].imshow(img)
    axes[0, 0].set_title('1. Original')
    axes[0, 0].axis('off')
    
    resultados_totales = {}
    
    # PASO 4: Procesar cada color
    col_idx = 1
    for nombre, h_range, s_range, v_range in colores_definidos:
        
        mask_h = (img_hsv[:, :, 0] >= h_range[0]) & (img_hsv[:, :, 0] <= h_range[1])
        mask_s = (img_hsv[:, :, 1] >= s_range[0]) & (img_hsv[:, :, 1] <= s_range[1])
        mask_v = (img_hsv[:, :, 2] >= v_range[0]) & (img_hsv[:, :, 2] <= v_range[1])
        mascara = mask_h & mask_s & mask_v
        mascara_uint8 = mascara.astype(np.uint8) * 255
        
        etiquetas = label(mascara)
        num_regiones = etiquetas.max()
        regiones = regionprops(etiquetas)
        
        resultados_totales[nombre] = {
            'mascara': mascara_uint8,
            'num': num_regiones,
            'regiones': regiones
        }
        
        axes[0, col_idx].imshow(mascara_uint8, cmap='gray')
        axes[0, col_idx].set_title(f'{nombre}\n{num_regiones} regiones')
        axes[0, col_idx].axis('off')
        
        print(f'PASO 4: {nombre} = {num_regiones} regiones')
        col_idx += 1
    
    axes[0, 4].axis('off')
    
    # PASO 5: Grises
    gris = rgb2gray(img)
    axes[1, 0].imshow(gris, cmap='gray')
    axes[1, 0].set_title('5. Grises')
    axes[1, 0].axis('off')
    
    # PASO 6: Suavizar
    gris_suave = gaussian(gris, sigma=1.0)
    axes[1, 1].imshow(gris_suave, cmap='gray')
    axes[1, 1].set_title('6. Suavizado')
    axes[1, 1].axis('off')
    
    # PASO 7: Umbralizar
    umbral = threshold_otsu(gris_suave)
    binaria = gris_suave > umbral
    axes[1, 2].imshow(binaria, cmap='gray')
    axes[1, 2].set_title(f'7. Binaria (u={umbral:.2f})')
    axes[1, 2].axis('off')
    
    # PASO 8: Etiquetar todas las regiones
    etiquetas_todas = label(binaria)
    axes[1, 3].imshow(etiquetas_todas, cmap='nipy_spectral')
    axes[1, 3].set_title(f'8. Todas ({etiquetas_todas.max()})')
    axes[1, 3].axis('off')
    
    # PASO 9: Obtener propiedades
    todas_regiones = regionprops(etiquetas_todas)
    
    # PASO 10: Analizar color por región
    datos_todos = []
    for i, reg in enumerate(todas_regiones):
        y, x = reg.bbox[0], reg.bbox[1]
        y2, x2 = reg.bbox[2], reg.bbox[3]
        
        conteo_colores = {}
        for nombre, datos in resultados_totales.items():
            region_mascara = datos['mascara'][y:y2, x:x2]
            conteo_colores[nombre] = np.sum(region_mascara > 0)
        
        color_predominante = max(conteo_colores, key=conteo_colores.get)
        
        datos_todos.append({
            'region': i + 1,
            'area': reg.area,
            'color': color_predominante,
            'pixeles_color': conteo_colores[color_predominante]
        })
    
    axes[1, 4].axis('off')
    
    # PASO 11: Resumen
    axes[2, 0].axis('off')
    axes[2, 1].axis('off')
    axes[2, 2].axis('off')
    axes[2, 3].axis('off')
    axes[2, 4].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # Mostrar tabla
    df = pd.DataFrame(datos_todos)
    print('\n=== RESUMEN DE REGIONES ===')
    print(df.to_string(index=False))
    
    print('\n=== CONTEO TOTAL ===')
    for nombre in resultados_totales.keys():
        cantidad = sum(1 for d in datos_todos if d['color'] == nombre)
        print(f'{nombre}: {cantidad} regiones')
    
    return resultados_totales, datos_todos


# EJECUTAR SPAGHETTI CON image.png
print('\n' + '=' * 50)
print('DIVIDIR Y CONTAR POR COLOR')
print('=' * 50 + '\n')

res = dividir_y_contar_por_color(RUTA_IMAGEN)


# EJEMPLO: Crear imagen de prueba si no existe image.png

In [ ]:
# Crear imagen de prueba si no existe
if not os.path.exists(RUTA_IMAGEN):
    print('Creando imagen de prueba...')
    np.random.seed(42)
    img_test = np.zeros((300, 400, 3))
    img_test[:] = [0.3, 0.3, 0.3]
    
    def dibujar_circulo(img, cx, cy, radio, color_rgb):
        y, x = np.ogrid[-radio:radio, -radio:radio]
        mask = x*x + y*y <= radio*radio
        img[cy-radio:cy+radio, cx-radio:cx+radio][mask] = color_rgb
        return img
    
    for _ in range(6):
        cx, cy = np.random.randint(30, 370), np.random.randint(30, 270)
        radio = np.random.randint(20, 35)
        img_test = dibujar_circulo(img_test, cx, cy, radio, [1, 0, 0])
    
    for _ in range(4):
        cx, cy = np.random.randint(30, 370), np.random.randint(30, 270)
        radio = np.random.randint(20, 35)
        img_test = dibujar_circulo(img_test, cx, cy, radio, [0, 0, 1])
    
    for _ in range(5):
        cx, cy = np.random.randint(30, 370), np.random.randint(30, 270)
        radio = np.random.randint(20, 35)
        img_test = dibujar_circulo(img_test, cx, cy, radio, [0, 1, 0])
    
    io.imsave(RUTA_IMAGEN, (img_test * 255).astype(np.uint8))
    print(f'Imagen guardada en {RUTA_IMAGEN}')
else:
    print(f'Usando imagen existente: {RUTA_IMAGEN}')
